# 01 — Download ERA5 MSL for Storm Claudia

Downloads a ±`window_days` window of global 6-hourly MSL at 2.5° from CDS
around the storm naming date. Global domain is required for spectral filtering
in the tracking step.

In [ ]:
import datetime as dt
import pathlib
import tempfile

import cdsapi
import xarray as xr

In [ ]:
storm_name = "claudia"
naming_date_str = "2025-11-10"
window_days = 7
output_file = "data/raw/msl_claudia_7d_global_2p5deg.nc"

In [ ]:
# process params
output_path = pathlib.Path(output_file)

naming_date = dt.datetime.fromisoformat(naming_date_str)
window_start = naming_date - dt.timedelta(days=window_days)
window_end = naming_date + dt.timedelta(days=window_days)
print(f"{storm_name.capitalize()}: {window_start.date()} → {window_end.date()}")

_days = [
    window_start + dt.timedelta(days=i)
    for i in range((window_end - window_start).days + 1)
]
cds = cdsapi.Client()
with tempfile.NamedTemporaryFile(suffix=".nc") as _tmp:
    cds.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": "mean_sea_level_pressure",
            "year": sorted({str(d.year) for d in _days}),
            "month": sorted({f"{d.month:02d}" for d in _days}),
            "day": sorted({f"{d.day:02d}" for d in _days}),
            "time": ["00:00", "06:00", "12:00", "18:00"],
            "grid": "2.5/2.5",
            "format": "netcdf",
        },
        _tmp.name,
    )
    _ds = xr.open_dataset(_tmp.name)
    _time_dim = "valid_time" if "valid_time" in _ds.dims else "time"
    _msl = _ds["msl"].sel({_time_dim: slice(window_start, window_end)})
    _msl = _msl.rename({_time_dim: "time"})
    _msl.to_netcdf(output_path)

print(f"Saved → {output_path}  ({output_path.stat().st_size / 1e6:.1f} MB)")